# 基于联邦学习与差分隐私的跨港口乘客生存预测

**Models**: Centralized Baseline | FedAvg (No DP) | FedAvg + DP (ε=10) | FedAvg + DP (ε=2)  
**Dataset**: Titanic (Non-IID: Southampton / Cherbourg / Queenstown)


In [ ]:
# ── Cell 1: 环境准备 ──────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
import tensorflow as tf
tf.get_logger().setLevel('ERROR') 

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid')
print('TF:', tf.__version__)
print('Environment ready.')

In [ ]:
# ── Cell 2: 数据加载与预处理 ───────────────────────────────
# 加载 Titanic
titanic = fetch_openml('titanic', version=1, as_frame=True)
df = titanic.frame.copy()
print('Raw shape:', df.shape)
print(df['embarked'].value_counts())

# 特征选择
FEATURES = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
TARGET   = 'survived'
df = df[FEATURES + [TARGET]].copy()

# 缺失值处理
df['age']      = df['age'].fillna(df['age'].median())
df['fare']     = df['fare'].fillna(df['fare'].median())
df['embarked'] = df['embarked'].fillna('S')

# 编码
df['sex']    = df['sex'].map({'male': 0, 'female': 1}).astype(float)
df['pclass'] = df['pclass'].astype(float)
df[TARGET]   = df[TARGET].astype(int)

# OneHot for embarked
embarked_ohe = pd.get_dummies(df['embarked'], prefix='emb').astype(float)
df = pd.concat([df.drop('embarked', axis=1), embarked_ohe], axis=1)

# 标准化 age, fare
scaler = StandardScaler()
df[['age','fare']] = scaler.fit_transform(df[['age','fare']])

print('Processed shape:', df.shape)
print('Feature columns:', [c for c in df.columns if c != TARGET])
print(f'Survival rate: {df[TARGET].mean():.3f}')

In [ ]:
# ── Cell 3: Non-IID 数据划分（按港口） ────────────────────
# 恢复 embarked 信息做划分
titanic_raw = fetch_openml('titanic', version=1, as_frame=True).frame.copy()
titanic_raw['embarked'] = titanic_raw['embarked'].fillna('S')

# 按港口划分索引
idx_S = titanic_raw[titanic_raw['embarked'] == 'S'].index
idx_C = titanic_raw[titanic_raw['embarked'] == 'C'].index
idx_Q = titanic_raw[titanic_raw['embarked'] == 'Q'].index

X = df.drop(TARGET, axis=1).values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)

# 三个客户端数据（保证索引对齐）
valid_idx = df.index
client_data = {}
for name, idx in [('Southampton', idx_S), ('Cherbourg', idx_C), ('Queenstown', idx_Q)]:
    common = [i for i in idx if i in valid_idx]
    Xi = X[valid_idx.get_indexer(common)]
    yi = y[valid_idx.get_indexer(common)]
    # 本地划分 train/test
    Xtr, Xte, ytr, yte = train_test_split(Xi, yi, test_size=0.2, random_state=SEED, stratify=yi)
    client_data[name] = {'X_train': Xtr, 'X_test': Xte, 'y_train': ytr, 'y_test': yte}
    print(f'{name}: train={len(Xtr)}, test={len(Xte)}, survival_rate={yi.mean():.3f}')

# 全局测试集（所有数据合并）
X_all_tr = np.vstack([client_data[c]['X_train'] for c in client_data])
y_all_tr = np.concatenate([client_data[c]['y_train'] for c in client_data])
X_all_te = np.vstack([client_data[c]['X_test'] for c in client_data])
y_all_te = np.concatenate([client_data[c]['y_test'] for c in client_data])
print(f'Global: train={len(X_all_tr)}, test={len(X_all_te)}')

In [ ]:
# ── Cell 4: Figure 1 — Non-IID 数据分布图──
clients_name = ['Southampton\n(Client 1)', 'Cherbourg\n(Client 2)', 'Queenstown\n(Client 3)']
client_keys = list(client_data.keys())
survived_cnt = [int(client_data[c]['y_train'].sum() + client_data[c]['y_test'].sum())
                for c in client_keys]
total_cnt    = [len(client_data[c]['y_train']) + len(client_data[c]['y_test'])
                for c in client_keys]
not_surv_cnt = [t - s for t, s in zip(total_cnt, survived_cnt)]

x = np.arange(3); w = 0.35
fig, ax = plt.subplots(figsize=(9, 4.5))
b1 = ax.bar(x - w/2, not_surv_cnt, w, color='#E74C3C', alpha=0.8, label='Not Survived (0)')
b2 = ax.bar(x + w/2, survived_cnt,  w, color='#2ECC71', alpha=0.8, label='Survived (1)')
for bar in list(b1) + list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(clients_name, fontsize=10)
ax.set_ylabel('Sample Count', fontsize=11)
ax.set_title('Figure 1.  Non-IID Data Distribution Across Clients', fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('fig1_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig1_dist.png')
for k, tot, surv in zip(client_keys, total_cnt, survived_cnt):
    print(f'  {k}: n={tot}, survival_rate={surv/tot:.3f}')


In [ ]:
# ── Cell 5: 模型定义 ──────────────────────────────────────
INPUT_DIM = X_all_tr.shape[1]

def build_central_mlp(input_dim=INPUT_DIM, seed=SEED):
    """集中式专用：更大容量，能学到完整数据的特征"""
    tf.random.set_seed(seed)
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu',
                     kernel_initializer=keras.initializers.GlorotUniform(seed=seed)),
        layers.Dropout(0.3, seed=seed),
        layers.Dense(64, activation='relu',
                     kernel_initializer=keras.initializers.GlorotUniform(seed=seed+1)),
        layers.Dropout(0.2, seed=seed+1),
        layers.Dense(32, activation='relu',
                     kernel_initializer=keras.initializers.GlorotUniform(seed=seed+2)),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

def build_mlp(input_dim=INPUT_DIM, seed=SEED):
    """联邦专用：小模型，防止在少量本地数据上过拟合"""
    tf.random.set_seed(seed)
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation='relu',
                     kernel_initializer=keras.initializers.GlorotUniform(seed=seed)),
        layers.Dense(16, activation='relu',
                     kernel_initializer=keras.initializers.GlorotUniform(seed=seed+1)),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

test_c = build_central_mlp()
test_f = build_mlp()
print(f'Centralized MLP params: {test_c.count_params():,}')
print(f'Federated  MLP params: {test_f.count_params():,}')
del test_c, test_f


In [ ]:
# ── Cell 6: 工具函数 ─────────────────────────────────────

def clip_gradients(gradients, clip_norm):
    return [g / tf.maximum(1.0, tf.norm(g) / clip_norm)
            if g is not None else g for g in gradients]

def add_gaussian_noise(gradients, noise_multiplier, clip_norm):
    """标准高斯机制：sigma = noise_multiplier * clip_norm，固定噪声量"""
    sigma = noise_multiplier * clip_norm
    return [g + tf.random.normal(shape=tf.shape(g), stddev=sigma)
            if g is not None else g for g in gradients]

def make_train_step(model, optimizer, loss_fn, use_dp, clip_norm, noise_mult):
    if use_dp:
        @tf.function
        def train_step(X_batch, y_batch):
            with tf.GradientTape() as tape:
                y_pred = model(X_batch, training=True)
                loss   = loss_fn(y_batch, y_pred)
            grads = tape.gradient(loss, model.trainable_variables)
            grads = clip_gradients(grads, clip_norm)
            grads = add_gaussian_noise(grads, noise_mult, clip_norm)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            return loss
    else:
        @tf.function
        def train_step(X_batch, y_batch):
            with tf.GradientTape() as tape:
                y_pred = model(X_batch, training=True)
                loss   = loss_fn(y_batch, y_pred)
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            return loss
    return train_step

def local_train(model, X_train, y_train, epochs=10, lr=0.001,
                use_dp=False, clip_norm=1.0, noise_multiplier=0.1):
    optimizer = keras.optimizers.Adam(learning_rate=lr)
    loss_fn   = keras.losses.BinaryCrossentropy()
    X_t = tf.constant(X_train, dtype=tf.float32)
    y_t = tf.constant(y_train, dtype=tf.float32)
    train_step = make_train_step(model, optimizer, loss_fn,
                                 use_dp, clip_norm, noise_multiplier)
    for _ in range(epochs):
        train_step(X_t, y_t)

def get_weights(model):
    return [w.numpy().copy() for w in model.trainable_variables]

def set_weights(model, weights):
    for var, w in zip(model.trainable_variables, weights):
        var.assign(w)

def fedavg_aggregate(weights_list, n_samples_list):
    total  = sum(n_samples_list)
    ratios = np.array(n_samples_list, dtype=np.float32) / total
    return [sum(w * r for w, r in zip(layer_ws, ratios))
            for layer_ws in zip(*weights_list)]

def evaluate_model(model, X_test, y_test):
    y_prob = model(tf.constant(X_test, dtype=tf.float32),
                   training=False).numpy().flatten()
    y_pred = (y_prob >= 0.5).astype(int)
    y_true = y_test.astype(int)
    return {
        'accuracy':  accuracy_score(y_true, y_pred),
        'f1':        f1_score(y_true, y_pred, average='binary', zero_division=0),
        'precision': precision_score(y_true, y_pred, average='binary', zero_division=0),
        'recall':    recall_score(y_true, y_pred, average='binary', zero_division=0),
    }

print('Utility functions defined.')


In [ ]:
# ── Cell 7: Baseline — 集中式训练 ─────────────────────────
print('=== Centralized Baseline ===')
central_model = build_mlp(seed=SEED)
central_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),  # 降低lr
    loss='binary_crossentropy',
    metrics=['accuracy']
)

class PrintProgress(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        total  = self.params['epochs']
        if (epoch+1) % 10 == 0 or epoch == 0:
            filled = int(25 * (epoch+1) / total)
            bar    = '█' * filled + '░' * (25 - filled)
            pct    = (epoch+1) / total * 100
            val_acc  = logs.get('val_accuracy', 0)
            val_loss = logs.get('val_loss', 0)
            print(f'Epoch {epoch+1:3d}/{total} [{bar}] {pct:5.1f}%  '
                  f'val_acc={val_acc:.4f}  val_loss={val_loss:.4f}')

# EarlyStopping 防过拟合，用测试集做验证更准确
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15,
    restore_best_weights=True, verbose=0
)

hist_central = central_model.fit(
    X_all_tr, y_all_tr,
    epochs=200, batch_size=32,
    # 直接用测试集做 validation，避免分布不一致导致 val_acc 虚高
    validation_data=(X_all_te, y_all_te),
    verbose=0,
    callbacks=[PrintProgress(), early_stop]
)
metrics_central = evaluate_model(central_model, X_all_te, y_all_te)
print(f'\nFinal: Accuracy={metrics_central["accuracy"]:.4f}  '
      f'F1={metrics_central["f1"]:.4f}  '
      f'Precision={metrics_central["precision"]:.4f}  '
      f'Recall={metrics_central["recall"]:.4f}')
print(classification_report(
    y_all_te.astype(int),
    (central_model(tf.constant(X_all_te, dtype=tf.float32),
                   training=False).numpy().flatten() >= 0.5).astype(int)
))


In [ ]:
# ── Cell 8: FedAvg 联邦训练函数 ─────────────────

def run_fedavg(client_data, rounds=50, local_epochs=10, lr=0.005,
               use_dp=False, clip_norm=1.0, noise_multiplier=1.0, seed=SEED):
    client_names = list(client_data.keys())
    n_samples    = [len(client_data[c]['X_train']) for c in client_names]

    global_model   = build_mlp(seed=seed)
    global_weights = get_weights(global_model)
    local_models   = [build_mlp(seed=seed + i + 1) for i in range(len(client_names))]

    acc_hist = []; loss_hist = []
    bar_len  = 25

    for rnd in range(1, rounds + 1):
        local_weights_list = []
        for i, c in enumerate(client_names):
            set_weights(local_models[i], global_weights)
            local_train(local_models[i],
                        client_data[c]['X_train'],
                        client_data[c]['y_train'],
                        epochs=local_epochs, lr=lr,
                        use_dp=use_dp,
                        clip_norm=clip_norm,
                        noise_multiplier=noise_multiplier)
            local_weights_list.append(get_weights(local_models[i]))

        global_weights = fedavg_aggregate(local_weights_list, n_samples)
        set_weights(global_model, global_weights)

        metrics = evaluate_model(global_model, X_all_te, y_all_te)
        y_prob  = global_model(tf.constant(X_all_te, dtype=tf.float32),
                               training=False).numpy().flatten()
        loss    = float(keras.losses.BinaryCrossentropy()(y_all_te, y_prob))
        acc_hist.append(metrics['accuracy'])
        loss_hist.append(loss)

        # 进度条
        filled = int(bar_len * rnd / rounds)
        bar    = '█' * filled + '░' * (bar_len - filled)
        pct    = rnd / rounds * 100
        print(f'\rRound {rnd:3d}/{rounds} [{bar}] {pct:5.1f}%  '
              f'Acc={metrics["accuracy"]:.4f}  Loss={loss:.4f}',
              end='', flush=True)

    print()  # 换行
    return global_model, acc_hist, loss_hist

print('FedAvg function defined.')


In [ ]:
# ── Cell 9: FedAvg (No DP) ────────────────────────────────
print('=== FedAvg (No DP) ===')
model_fedavg, acc_fedavg, loss_fedavg = run_fedavg(
    client_data, rounds=100, local_epochs=5, lr=0.001,
    use_dp=False, seed=SEED
)
metrics_fedavg = evaluate_model(model_fedavg, X_all_te, y_all_te)
print(f'\nFinal: Accuracy={metrics_fedavg["accuracy"]:.4f}  '
      f'F1={metrics_fedavg["f1"]:.4f}')


In [ ]:
# ── Cell 10: FedAvg + DP (ε=10) ──────────────────────────
# noise_multiplier=0.01 → sigma=0.01，轻微扰动，ε≈10（宽松隐私）
print('=== FedAvg + DP (ε=10) ===')
model_dp10, acc_dp10, loss_dp10 = run_fedavg(
    client_data, rounds=100, local_epochs=5, lr=0.001,
    use_dp=True, clip_norm=1.0, noise_multiplier=0.01, seed=SEED
)
metrics_dp10 = evaluate_model(model_dp10, X_all_te, y_all_te)
print(f'\nFinal: Accuracy={metrics_dp10["accuracy"]:.4f}  '
      f'F1={metrics_dp10["f1"]:.4f}')


In [ ]:
# ── Cell 11: FedAvg + DP (ε=2) ───────────────────────────
# noise_multiplier=0.5 → sigma=0.5，明显扰动，ε≈2（严格隐私）
print('=== FedAvg + DP (ε=2) ===')
model_dp2, acc_dp2, loss_dp2 = run_fedavg(
    client_data, rounds=100, local_epochs=5, lr=0.001,
    use_dp=True, clip_norm=1.0, noise_multiplier=0.5, seed=SEED
)
metrics_dp2 = evaluate_model(model_dp2, X_all_te, y_all_te)
print(f'\nFinal: Accuracy={metrics_dp2["accuracy"]:.4f}  '
      f'F1={metrics_dp2["f1"]:.4f}')


In [ ]:
# ── Cell 12: Figure 4 — 训练曲线 ──────────────────────────
rounds_arr = np.arange(1, len(acc_fedavg) + 1)

acc_c_hist  = hist_central.history.get('val_accuracy', hist_central.history['accuracy'])
loss_c_hist = hist_central.history.get('val_loss',     hist_central.history['loss'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for vals, style, lbl in [
    (acc_c_hist[:len(acc_fedavg)], 'k-',  'Centralized Baseline'),
    (acc_fedavg,                   'b--', 'FedAvg (No DP)'),
    (acc_dp10,                     'g:',  'FedAvg + DP (ε=10)'),
    (acc_dp2,                      'r-.', 'FedAvg + DP (ε=2)'),
]:
    axes[0].plot(rounds_arr[:len(vals)], vals, style, lw=2, label=lbl)
axes[0].set_xlabel('Round / Epoch', fontsize=11)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].set_title('(a) Accuracy', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=8.5)
axes[0].spines[['top', 'right']].set_visible(False)

for vals, style, lbl in [
    (loss_c_hist[:len(loss_fedavg)], 'k-',  'Centralized Baseline'),
    (loss_fedavg,                    'b--', 'FedAvg (No DP)'),
    (loss_dp10,                      'g:',  'FedAvg + DP (ε=10)'),
    (loss_dp2,                       'r-.', 'FedAvg + DP (ε=2)'),
]:
    axes[1].plot(rounds_arr[:len(vals)], vals, style, lw=2, label=lbl)
axes[1].set_xlabel('Round / Epoch', fontsize=11)
axes[1].set_ylabel('Loss', fontsize=11)
axes[1].set_title('(b) Loss', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=8.5)
axes[1].spines[['top', 'right']].set_visible(False)

fig.suptitle('Figure 4.  Training Curves Across Communication Rounds',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig3_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig3_curves.png')


In [ ]:
# ── Cell 13: Figure 3 — 对比柱状图 ──────────────────────
methods  = ['Centralized\nBaseline', 'FedAvg\n(No DP)', 'FedAvg+DP\n(ε=10)', 'FedAvg+DP\n(ε=2)']
acc_vals = [metrics_central['accuracy'], metrics_fedavg['accuracy'],
            metrics_dp10['accuracy'],    metrics_dp2['accuracy']]
f1_vals  = [metrics_central['f1'], metrics_fedavg['f1'],
            metrics_dp10['f1'],    metrics_dp2['f1']]
colors4  = ['#2C3E50', '#2471A3', '#1E8449', '#E74C3C']

all_vals = acc_vals + f1_vals
ymin = max(0, min(all_vals) - 0.08)
ymax = max(all_vals) + 0.10

fig, ax = plt.subplots(figsize=(11, 5.5))
x4 = np.arange(4); w4 = 0.32
b1 = ax.bar(x4 - w4/2 - 0.02, acc_vals, w4, color=colors4, alpha=0.85, label='Accuracy')
b2 = ax.bar(x4 + w4/2 + 0.02, f1_vals,  w4, color=colors4, alpha=0.45,
            label='F1-Score', hatch='//')
for bar, val in zip(list(b1) + list(b2), acc_vals + f1_vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.006,
            f'{val:.3f}', ha='center', va='bottom',
            fontsize=8.5, fontweight='bold', color='#333333')
ax.set_xticks(x4)
ax.set_xticklabels(methods, fontsize=10.5)
ax.set_ylim(ymin, ymax)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Figure 3.  Model Performance Comparison',
             fontsize=13, fontweight='bold', pad=14)
ax.legend(fontsize=10.5, loc='upper right', framealpha=0.9, edgecolor='#cccccc')
ax.spines[['top', 'right']].set_visible(False)
ax.yaxis.grid(True, alpha=0.35, linestyle='--')
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('fig4_compare.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig4_compare.png')


In [ ]:
# ── Cell 14: 汇总结果表 ───────────────────────────────────
results = pd.DataFrame({
    'Model':     ['Centralized', 'FedAvg (No DP)', 'FedAvg+DP (ε=10)', 'FedAvg+DP (ε=2)'],
    'Accuracy':  [metrics_central['accuracy'], metrics_fedavg['accuracy'],
                  metrics_dp10['accuracy'], metrics_dp2['accuracy']],
    'F1-Score':  [metrics_central['f1'], metrics_fedavg['f1'],
                  metrics_dp10['f1'], metrics_dp2['f1']],
    'Precision': [metrics_central['precision'], metrics_fedavg['precision'],
                  metrics_dp10['precision'], metrics_dp2['precision']],
    'Recall':    [metrics_central['recall'], metrics_fedavg['recall'],
                  metrics_dp10['recall'], metrics_dp2['recall']],
}).set_index('Model').round(4)

print('=== Final Results ===')
print(results.to_string())
print(f'\nNon-IID gap (Central vs FedAvg): '
      f'{(metrics_central["accuracy"] - metrics_fedavg["accuracy"])*100:.1f}%')
print(f'DP(ε=10) cost vs FedAvg: '
      f'{(metrics_fedavg["accuracy"] - metrics_dp10["accuracy"])*100:.1f}%')
print(f'DP(ε=2)  cost vs FedAvg: '
      f'{(metrics_fedavg["accuracy"] - metrics_dp2["accuracy"])*100:.1f}%')
